In [2]:
import pandas as pd 
import numpy as np

In [3]:
#path to eZTrack data 

ezTrackLocations = r'/Users/johnmarshall/Documents/Analysis/miniscope_analysis/miniscopeLinearTrack/2025.4/m328/2025_04_10_328_15_59_58_b2/My_WebCam/2025_04_10_328_15_59_58_b2_rotated_cropped_output_LocationOutput.csv'
ezTrack = alignedTraces = pd.read_csv(ezTrackLocations)

#path to aligned eZTrack data
dirPath = '/Users/johnmarshall/Documents/Analysis/miniscope_analysis/miniscopeLinearTrack/2025.4/m328/2025_04_10_328_15_59_58_b2/'
alignedFile = '1_24_motion_corrected_JJM_5_9_25cellTracesAlignedToTracking.csv'
#load calcium signal aligned to tracking data 
alignedTraces = pd.read_csv(dirPath+alignedFile)

In [4]:
alignedTracesLocationData = alignedTraces[['closestBehavCamFrameIdx', 'X_coor', 'Y_coor']]

In [5]:
ezTrack[['X', 'Y']][0:50].head()

,X,Y
0,646.160325,20.247228
1,645.131117,19.839175
2,645.342843,19.225590
3,644.350702,18.985817
4,643.659900,18.418524


In [6]:
alignedTracesLocationData[0:50].head()

,closestBehavCamFrameIdx,X_coor,Y_coor
0,0,646.160325,20.247228
1,1,645.131117,19.839175
2,2,645.342843,19.225590
3,3,644.350702,18.985817
4,4,643.659900,18.418524


In [ ]:
##validate with video overlay  
#small video 

# ── CONFIG ─────────────────────────────────────────────────────────────────────

dirPath      = '/Users/johnmarshall/Documents/Analysis/miniscope_analysis/miniscopeLinearTrack/2025.4/m328/2025_04_10_328_15_59_58_b2/My_WebCam/'
rotatedVideo = 'video_output.avi'
input_vid    = os.path.join(dirPath, rotatedVideo)
output_vid   = os.path.join(dirPath, 'with_velocity_overlay.mp4')

# your velocity array (length == # frames)
vel_array = binned['velocity2dSpatialFiltered'].to_numpy()

# your per-behavior-frame X positions (make sure it's indexed by frame 0…n-1)
# e.g. binned = binned.reset_index().set_index('frame')
x_pos = binned['X_coor'].to_numpy()

# height of the black bar you want to add
bar_h = 50

# ── SET UP I/O ─────────────────────────────────────────────────────────────────

cap      = cv2.VideoCapture(input_vid)
fps      = cap.get(cv2.CAP_PROP_FPS)
w        = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
h        = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
n_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

# choose a codec that FIJI likes for AVIs
fourcc = cv2.VideoWriter_fourcc(*'XVID')
out    = cv2.VideoWriter(output_vid, fourcc, fps, (w, h + bar_h))

# text & dot styling
font        = cv2.FONT_HERSHEY_SIMPLEX
font_scale  = 1
thickness   = 2
margin      = 10
dot_radius  = 5

# ── PROCESS FRAMES ──────────────────────────────────────────────────────────────

for frame_idx in tqdm(range(n_frames), desc="Adding overlay"):
    ret, frame = cap.read()
    if not ret:
        break

    # 1) create the padded frame
    bar       = np.zeros((bar_h, w, 3), dtype=np.uint8)
    new_frame = np.vstack((frame, bar))

    # 2) overlay the velocity text in the bar (left‐aligned)
    vel_text = f"{vel_array[frame_idx-1]:.2f} cm/s"
    (tw, th), _ = cv2.getTextSize(vel_text, font, font_scale, thickness)
    text_x = margin
    text_y = h + margin + th
    cv2.putText(
        new_frame,
        vel_text,
        (text_x, text_y),
        font,
        font_scale,
        (255, 255, 255),
        thickness,
        cv2.LINE_AA
    )

    # 3) draw the dot at the mouse’s X in the center of the bar
    xm = int(np.clip(x_pos[frame_idx-1], 0, w-1))
    y_dot = h + bar_h // 2
    cv2.circle(new_frame, (xm, y_dot), dot_radius, (255, 255, 255), -1)

    out.write(new_frame)

# ── CLEAN UP ────────────────────────────────────────────────────────────────────

cap.release()
out.release()
print("Done – saved with overlay to:", output_vid)